In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

def prepare_diabetes_data(filepath):
    df = pd.read_csv(filepath)

    # Standardize missing markers
    df = df.replace("?", np.nan)

    # Drop obvious identifier columns / very high-cardinality IDs
    drop_cols = [
        "encounter_id",
        "patient_nbr",
    ]
    existing_drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=existing_drop_cols)

    # Optional: remove weight / payer_code / medical_specialty if too sparse
    sparse_optional = ["weight", "payer_code", "medical_specialty"]
    sparse_existing = [c for c in sparse_optional if c in df.columns]
    df = df.drop(columns=sparse_existing)

    # Create binary target: readmitted within 30 days vs not
    # Common values are '<30', '>30', 'NO'
    if "readmitted" not in df.columns:
        raise ValueError("Expected column 'readmitted' not found in diabetes dataset.")

    df["target"] = (df["readmitted"] == "<30").astype(int)
    df = df.drop(columns=["readmitted"])

    # Convert some numeric-looking columns if present
    numeric_candidates = [
        "time_in_hospital", "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient", "number_emergency",
        "number_inpatient", "number_diagnoses"
    ]
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    x = df.drop(columns=["target"])
    y = df["target"]

    return x, y

def prepare_german_credit_data(filepath):
    import pandas as pd

    col_names = [
        "status", "duration", "credit_history", "purpose", "credit_amount",
        "savings", "employment", "installment_rate", "personal_status_sex",
        "other_debtors", "residence_since", "property", "age",
        "other_installment_plans", "housing", "existing_credits",
        "job", "num_dependents", "telephone", "foreign_worker", "target"
    ]

    df = pd.read_csv(filepath, sep=" ", header=None, names=col_names)

    # Convert target: 1 = good, 2 = bad → make 1 = bad
    df["target"] = (df["target"] == 2).astype(int)

    # Extract gender for fairness analysis
    def extract_gender(val):
        if val in ["A91", "A93", "A94"]:
            return "male"
        elif val in ["A92", "A95"]:
            return "female"
        else:
            return "unknown"

    df["gender"] = df["personal_status_sex"].apply(extract_gender)

    x = df.drop(columns=["target"])
    y = df["target"]

    return x, y

def prepare_synthetic_diabetes_data(filepath):
    df = pd.read_csv(filepath)

    # Standardize missing markers
    df = df.replace("?", np.nan)

    # Drop identifier / sparse columns to match real-data prep
    drop_cols = ["encounter_id", "patient_nbr"]
    existing_drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=existing_drop_cols)

    sparse_optional = ["weight", "payer_code", "medical_specialty"]
    sparse_existing = [c for c in sparse_optional if c in df.columns]
    df = df.drop(columns=sparse_existing)

    # Recreate binary target if needed
    if "target" not in df.columns:
        if "readmitted" not in df.columns:
            raise ValueError("Synthetic diabetes file must contain either 'target' or 'readmitted'.")
        df["target"] = (df["readmitted"] == "<30").astype(int)
        df = df.drop(columns=["readmitted"])
    else:
        df["target"] = pd.to_numeric(df["target"], errors="coerce").fillna(0).astype(int)

    numeric_candidates = [
        "time_in_hospital", "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient", "number_emergency",
        "number_inpatient", "number_diagnoses"
    ]
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

def prepare_synthetic_german_credit_data(filepath):
    df = pd.read_csv(filepath)

    # 🔥 Rename columns to match real dataset
    rename_map = {
        "checking_status": "status",
        "savings_status": "savings",
        "personal_status": "personal_status_sex",
        "installment_commitment": "installment_rate",
        "other_parties": "other_debtors",
        "property_magnitude": "property",
        "other_payment_plans": "other_installment_plans",
        "own_telephone": "telephone"
    }

    df = df.rename(columns=rename_map)

    # Handle target
    if "target" in df.columns:
        unique_vals = set(df["target"].dropna().unique())
        if unique_vals.issubset({1, 2}):
            df["target"] = (df["target"] == 2).astype(int)
        else:
            df["target"] = pd.to_numeric(df["target"], errors="coerce").fillna(0).astype(int)

    elif "class" in df.columns:
        df["target"] = (df["class"] == 2).astype(int)
        df = df.drop(columns=["class"])

    else:
        raise ValueError("No target column found.")

    # Recreate gender
    if "gender" not in df.columns and "personal_status_sex" in df.columns:
        def extract_gender(val):
            if val in ["A91", "A93", "A94"]:
                return "male"
            elif val in ["A92", "A95"]:
                return "female"
            else:
                return "unknown"

        df["gender"] = df["personal_status_sex"].apply(extract_gender)

    return df

def build_preprocessor(X):
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numerical_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numerical_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    return preprocessor, numerical_cols, categorical_cols


def build_logistic_model(preprocessor):
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            solver="liblinear",   # stable for many binary tabular problems
            class_weight="balanced"
        ))
    ])
    return model


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }

    if y_prob is not None and len(np.unique(y_test)) > 1:
        results["auc"] = roc_auc_score(y_test, y_prob)
    else:
        results["auc"] = np.nan

    return results, y_pred, y_prob


def fairness_by_group(model, X_test, y_test, group_col):
    if group_col not in X_test.columns:
        raise ValueError(f"Column '{group_col}' not found in X_test.")

    temp = X_test.copy()
    temp["y_true"] = y_test.values
    temp["y_pred"] = model.predict(X_test)

    rows = []

    for group_value in temp[group_col].dropna().unique():
        g = temp[temp[group_col] == group_value]

        y_true_g = g["y_true"]
        y_pred_g = g["y_pred"]

        # confusion matrix in fixed order [0,1]
        tn, fp, fn, tp = confusion_matrix(
            y_true_g, y_pred_g, labels=[0, 1]
        ).ravel()

        positive_rate = (y_pred_g == 1).mean()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

        rows.append({
            "group_column": group_col,
            "group_value": group_value,
            "n": len(g),
            "positive_rate": positive_rate,
            "FPR": fpr,
            "FNR": fnr,
            "precision": precision
        })

    return pd.DataFrame(rows)

def run_real_baseline(X, y, dataset_name):

    X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
        X, y,
        test_size=0.5,
        random_state=42,
        stratify=y
    )

    preprocessor, numerical_cols, categorical_cols = build_preprocessor(X_train_real)
    model = build_logistic_model(preprocessor)

    model.fit(X_train_real, y_train_real)
    metrics, y_pred, y_prob = evaluate_model(model, X_test_real, y_test_real)

    print(f"\n=== {dataset_name}: REAL baseline ===")
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("F1:", round(metrics["f1"], 4))
    print("AUC:", round(metrics["auc"], 4) if pd.notnull(metrics["auc"]) else "nan")

    return {
        "model": model,
        "X_train_real": X_train_real,
        "X_test_real": X_test_real,
        "y_train_real": y_train_real,
        "y_test_real": y_test_real,
        "metrics": metrics,
        "numerical_cols": numerical_cols,
        "categorical_cols": categorical_cols
    }


def run_synthetic_tstr(
    synthetic_train_df,
    X_test_real,
    y_test_real,
    target_col="target"
):

    if target_col not in synthetic_train_df.columns:
        raise ValueError(f"Synthetic dataframe must contain target column '{target_col}'.")

    X_train_synth = synthetic_train_df.drop(columns=[target_col])
    y_train_synth = synthetic_train_df[target_col].astype(int)

    preprocessor, _, _ = build_preprocessor(X_train_synth)
    model = build_logistic_model(preprocessor)

    model.fit(X_train_synth, y_train_synth)
    metrics, y_pred, y_prob = evaluate_model(model, X_test_real, y_test_real)

    print("\n=== Synthetic Train / Real Test ===")
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("F1:", round(metrics["f1"], 4))
    print("AUC:", round(metrics["auc"], 4) if pd.notnull(metrics["auc"]) else "nan")

    return {
        "model": model,
        "metrics": metrics
    }


In [3]:
X_diab, y_diab = prepare_diabetes_data("diabetic_data.csv")

diab_results = run_real_baseline(
    X=X_diab,
    y=y_diab,
    dataset_name="UCI Diabetes"
)


=== UCI Diabetes: REAL baseline ===
Accuracy: 0.6469
F1: 0.2479
AUC: 0.6282


In [4]:
X_credit, y_credit = prepare_german_credit_data("german.data")

credit_results = run_real_baseline(
    X=X_credit,
    y=y_credit,
    dataset_name="German Credit"
)


=== German Credit: REAL baseline ===
Accuracy: 0.694
F1: 0.5942
AUC: 0.7736


In [7]:
# Load synthetic datasets
diab_synth_df = prepare_synthetic_diabetes_data("diabetic_data_synthetic_gen1.csv")
credit_synth_df = prepare_synthetic_german_credit_data("german_credit_synthetic_gen1.csv")

# Run TSTR experiments
diab_tstr = run_synthetic_tstr(
    synthetic_train_df=diab_synth_df,
    X_test_real=diab_results["X_test_real"],
    y_test_real=diab_results["y_test_real"],
    target_col="target"
)

credit_tstr = run_synthetic_tstr(
    synthetic_train_df=credit_synth_df,
    X_test_real=credit_results["X_test_real"],
    y_test_real=credit_results["y_test_real"],
    target_col="target"
)


=== Synthetic Train / Real Test ===
Accuracy: 0.6073
F1: 0.2402
AUC: 0.6169

=== Synthetic Train / Real Test ===
Accuracy: 0.452
F1: 0.2751
AUC: 0.3738


In [8]:
def add_performance_result(rows, dataset_name, model_name, metrics_dict, test_data="Real"):
    rows.append({
        "Dataset": dataset_name,
        "Model": "Logistic Regression",
        "Train Data": model_name,
        "Test Data": test_data,
        "Accuracy": round(metrics_dict["accuracy"], 4),
        "F1": round(metrics_dict["f1"], 4),
        "AUC": round(metrics_dict["auc"], 4) if pd.notnull(metrics_dict["auc"]) else None
    })

def add_fairness_result(rows, dataset_name, train_data_name, fairness_df):
    for _, row in fairness_df.iterrows():
        rows.append({
            "Dataset": dataset_name,
            "Model": "Logistic Regression",
            "Train Data": train_data_name,
            "Group Column": row["group_column"],
            "Group Value": row["group_value"],
            "N": int(row["n"]),
            "Positive Rate": round(row["positive_rate"], 4) if pd.notnull(row["positive_rate"]) else None,
            "FPR": round(row["FPR"], 4) if pd.notnull(row["FPR"]) else None,
            "FNR": round(row["FNR"], 4) if pd.notnull(row["FNR"]) else None,
            "Precision": round(row["precision"], 4) if pd.notnull(row["precision"]) else None
        })


In [12]:
performance_rows = []

add_performance_result(performance_rows, "UCI Diabetes", "Real", diab_results["metrics"])
add_performance_result(performance_rows, "UCI Diabetes", "Synthetic", diab_tstr["metrics"])

add_performance_result(performance_rows, "German Credit", "Real", credit_results["metrics"])
add_performance_result(performance_rows, "German Credit", "Synthetic", credit_tstr["metrics"])

performance_table = pd.DataFrame(performance_rows)
print(performance_table)

         Dataset                Model Train Data Test Data  Accuracy      F1  \
0   UCI Diabetes  Logistic Regression       Real      Real    0.6469  0.2479   
1   UCI Diabetes  Logistic Regression  Synthetic      Real    0.6073  0.2402   
2  German Credit  Logistic Regression       Real      Real    0.6940  0.5942   
3  German Credit  Logistic Regression  Synthetic      Real    0.4520  0.2751   

      AUC  
0  0.6282  
1  0.6169  
2  0.7736  
3  0.3738  


In [13]:
fairness_diab_real_gender = fairness_by_group(
    diab_results["model"],
    diab_results["X_test_real"],
    diab_results["y_test_real"],
    group_col="gender"
)

fairness_diab_real_race = fairness_by_group(
    diab_results["model"],
    diab_results["X_test_real"],
    diab_results["y_test_real"],
    group_col="race"
)

fairness_credit_real_gender = fairness_by_group(
    credit_results["model"],
    credit_results["X_test_real"],
    credit_results["y_test_real"],
    group_col="gender"
)

fairness_diab_synth_gender = fairness_by_group(
    diab_tstr["model"],
    diab_results["X_test_real"],
    diab_results["y_test_real"],
    group_col="gender"
)

fairness_diab_synth_race = fairness_by_group(
    diab_tstr["model"],
    diab_results["X_test_real"],
    diab_results["y_test_real"],
    group_col="race"
)

fairness_credit_synth_gender = fairness_by_group(
    credit_tstr["model"],
    credit_results["X_test_real"],
    credit_results["y_test_real"],
    group_col="gender"
)

fairness_rows = []

# REAL
add_fairness_result(fairness_rows, "UCI Diabetes", "Real", fairness_diab_real_gender)
add_fairness_result(fairness_rows, "UCI Diabetes", "Real", fairness_diab_real_race)
add_fairness_result(fairness_rows, "German Credit", "Real", fairness_credit_real_gender)

# SYNTHETIC
add_fairness_result(fairness_rows, "UCI Diabetes", "Synthetic", fairness_diab_synth_gender)
add_fairness_result(fairness_rows, "UCI Diabetes", "Synthetic", fairness_diab_synth_race)
add_fairness_result(fairness_rows, "German Credit", "Synthetic", fairness_credit_synth_gender)
fairness_table = pd.DataFrame(fairness_rows)
fairness_table["Group Value"] = fairness_table["Group Value"].str.lower()
print(fairness_table)

          Dataset                Model Train Data Group Column  \
0    UCI Diabetes  Logistic Regression       Real       gender   
1    UCI Diabetes  Logistic Regression       Real       gender   
2    UCI Diabetes  Logistic Regression       Real       gender   
3    UCI Diabetes  Logistic Regression       Real         race   
4    UCI Diabetes  Logistic Regression       Real         race   
5    UCI Diabetes  Logistic Regression       Real         race   
6    UCI Diabetes  Logistic Regression       Real         race   
7    UCI Diabetes  Logistic Regression       Real         race   
8   German Credit  Logistic Regression       Real       gender   
9   German Credit  Logistic Regression       Real       gender   
10   UCI Diabetes  Logistic Regression  Synthetic       gender   
11   UCI Diabetes  Logistic Regression  Synthetic       gender   
12   UCI Diabetes  Logistic Regression  Synthetic       gender   
13   UCI Diabetes  Logistic Regression  Synthetic         race   
14   UCI D